# 0. Setup

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

## 0.1 - Imports

In [ ]:
import os
import gc
import json
import random

import numpy as np
import pandas as pd

from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics.pairwise import cosine_similarity

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

from sentence_transformers import SentenceTransformer

from gensim.models import Word2Vec

import torch
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline,
    DataCollatorWithPadding
)

import wandb

from kaggle_secrets import UserSecretsClient

from huggingface_hub import login, whoami

import string

from transformers import get_linear_schedule_with_warmup

from torch.optim import AdamW

from torch.utils.data import Dataset, DataLoader
from collections import Counter

import torch.nn as nn

## 0.2 - Seeding for Reproducibility

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## 0.3 - Weights and Biases Setup

In [ ]:
user_secrets = UserSecretsClient()
wandb_api_key = user_secrets.get_secret("WANB_API_KEY")


wandb.login(
    key=wandb_api_key
)

## 0.4 - HuggingFace Setup (for Deploying)

In [ ]:
hf_token = user_secrets.get_secret(
    "hf_token"
)

login(hf_token)

whoami()

## 0.5 - Device Setup

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)
print(torch.cuda.get_device_name(0))

# 1. Preliminary Functions

## 1.1 - MAP@3 function

In [ ]:
# takes two inputs, one is a list of all correct answers, the other a list of corresponding predictions
def map_at_3(y_true, y_pred):
    score = 0.0

    if len(y_true) != len(y_pred):
        raise ValueError("Length mismatch")

    for i in range(len(y_true)):
        pred = y_pred[i]
        true = y_true[i]
        if true in pred:
            for j in range(len(pred)):
                if pred[j] == true:
                    score += 1/(j+1)
                    break
                
    return score/len(y_true)

## 1.2 - Text cleaning

In [ ]:
import re

def clean_text(text):

    text = str(text).lower()

    text = re.sub(
        r"[^a-z0-9 ]",
        " ",
        text
    )

    return text

## 1.3 - Expand Dataset
Creates question + answer pairs

In [ ]:
def expand_dataset(df):
    samples = []

    for _, row in df.iterrows():
        
        for option in ["A","B","C","D","E"]:
            samples.append({
                'prompt': row['prompt'],

                'option': row[option],
                
                "label": 1 if option == row["answer"] else 0,
    
                "question_id": row["id"]
            })

    return pd.DataFrame(samples)

## 1.4 - MCQDataset Class

class to create objects for DaBERTa models

In [ ]:
# create MCQDATASET class
class MCQDataset(Dataset):
    def __init__(self, dataframe):
        self.prompts = dataframe['prompt'].tolist()
        self.options = dataframe['option'].tolist()
        self.labels = dataframe['label'].tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        encoding = tokenizer(
            self.prompts[idx],
            self.options[idx],
            truncation=True,
            max_length=256
        )

        return {
            "input_ids":
                encoding['input_ids'],

            "attention_mask":
                encoding['attention_mask'],

            "labels": int(self.labels[idx])
        }

## 1.5 - Predict (for DaBERTa)

In [ ]:
import torch
import torch.nn.functional as F

OPTIONS = ["A", "B", "C", "D", "E"]

def predict_mcq(model, tokenizer, row, device):

    model.eval()

    scores = []

    with torch.no_grad():

        for option in OPTIONS:

            encoding = tokenizer(

                row["prompt"],
                row[option],

                truncation=True,
                padding="max_length",
                max_length=256,

                return_tensors="pt"

            )

            input_ids = encoding["input_ids"].to(device)
            attention_mask = encoding["attention_mask"].to(device)

            outputs = model(

                input_ids=input_ids,
                attention_mask=attention_mask

            )

            probabilities = F.softmax(outputs.logits, dim=1)

            score = probabilities[0,1].item()

            scores.append(score)

    ranking = [

        option

        for _, option in sorted(

            zip(scores, OPTIONS),

            reverse=True

        )

    ]

    return ranking, scores

# 2. Load Data

In [ ]:
df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
df

In [ ]:
df_test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
df_test

# 3 - Model - 1: BiLSTM

## 3.1 - Setup

### 3.1.1 - build_vocab

In [ ]:
def build_vocab(df):
    counter = Counter()

    for _,row in df.iterrows():
        counter.update(clean_text(row["prompt"]).split())

        for option in ["A", "B", "C", "D", "E"]:
            counter.update(clean_text(row[option]).split())

    vocab = {
        "<PAD>": 0,
        "<UNK>": 1
    }
    
    for word in counter:
        vocab[word] = len(vocab)

    return vocab

### 3.1.2 - Dataset Class

In [ ]:
class LSTMDataset(Dataset):
    def __init__(self, dataframe, vocab):
        self.df = expand_dataset(dataframe)

        self.vocab = vocab

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = row["prompt"] + " " + row["option"]

        return {

            "input_ids": torch.tensor(
                encode(text, self.vocab),
                dtype=torch.long
            ),

            "labels": torch.tensor(
                row["label"],
                dtype=torch.float
            )
        }

### 3.1.3 - Model Definition

In [ ]:
class BiLSTMClassifier(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim=300,
        hidden_dim=128
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.classifier = nn.Sequential(

            nn.Linear(hidden_dim * 2, 128),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(128, 1)

        )

    def forward(self, input_ids):

        x = self.embedding(input_ids)

        _, (hidden, _) = self.lstm(x)

        x = torch.cat(
            (hidden[-2], hidden[-1]),
            dim=1
        )

        return self.classifier(x).squeeze(1)

### 3.1.4 - Encode Function

In [ ]:
MAX_LEN = 128

def encode(text, vocab):

    tokens = clean_text(text).split()

    ids = [
        vocab.get(word, 1)
        for word in tokens
    ]

    ids = ids[:MAX_LEN]

    ids += [0] * (MAX_LEN - len(ids))

    return ids

## 3.2 - Training

### 3.2.1 - Load Data

In [ ]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df["answer"]
)

vocab = build_vocab(train_df)

train_dataset = LSTMDataset(train_df, vocab)
val_dataset = LSTMDataset(val_df, vocab)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32
)

### 3.2.2 - Model + criterion + optimizer

In [ ]:
model = BiLSTMClassifier(len(vocab)).to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

### 3.2.3 - Training Loop

In [ ]:
EPOCHS = 5

for epoch in range(EPOCHS):

    model.train()

    train_loss = 0

    for batch in train_loader:

        input_ids = batch["input_ids"].to(device)

        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        logits = model(input_ids)

        loss = criterion(logits, labels)

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    ########################################

    model.eval()

    predictions = []

    truths = []

    with torch.no_grad():

        for batch in val_loader:

            input_ids = batch["input_ids"].to(device)

            labels = batch["labels"]

            logits = model(input_ids)

            probs = torch.sigmoid(logits)

            preds = (probs > 0.5).int()

            predictions.extend(preds.cpu().numpy())

            truths.extend(labels.numpy())

    accuracy = accuracy_score(
        truths,
        predictions
    )

    print(
        f"Epoch {epoch+1} | "
        f"Loss {train_loss:.4f} | "
        f"Accuracy {accuracy:.4f}"
    )

## 3.3 - Inference

In [ ]:
OPTIONS = ["A", "B", "C", "D", "E"]

def predict_mcq_lstm(model, row, vocab):

    model.eval()

    scores = []

    with torch.no_grad():

        for option in OPTIONS:

            text = row["prompt"] + " " + row[option]

            ids = torch.tensor(

                encode(text, vocab),

                dtype=torch.long

            ).unsqueeze(0).to(device)

            logit = model(ids)

            score = torch.sigmoid(logit).item()

            scores.append(score)

    ranking = [

        option

        for _, option in sorted(

            zip(scores, OPTIONS),

            reverse=True

        )

    ]

    return ranking

In [ ]:
predictions = []

truths = []

for _, row in val_df.iterrows():

    predictions.append(

        predict_mcq_lstm(
            model,
            row,
            vocab
        )[:3]
    )

    truths.append(row["answer"])

print("MAP@3:", map_at_3(truths, predictions))

## 3.4 - prediction and submission

In [ ]:
test_predictions = []

for _, row in df_test.iterrows():

    pred = predict_mcq_lstm(
        model,
        row,
        vocab
    )[:3]

    test_predictions.append(pred)

In [ ]:
submission = pd.DataFrame({
    "ID": df_test["id"],
    "Prediction": [' '.join(pred) for pred in test_predictions]
})

submission.to_csv("submissionBiLSTM.csv", index=False)

submission.head()

# 4 - Model - 2: DaBerta - single

## 4.1 - Setup

### 4.1.1 - splitting dataset

In [ ]:
# Train/Validation split
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df['answer'],
    random_state=42
)

print(len(train_df))
print(len(val_df))

### 4.1.2 - expand dataset

In [ ]:
train_pairs = expand_dataset(train_df)

val_pairs = expand_dataset(val_df)

In [ ]:
train_pairs.head()

### 4.1.3 - Load Tokenizer

In [ ]:
# Load Deberta 
MODEL_NAME = "microsoft/deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

### 4.1.4 - Create Dataset Object

In [ ]:
train_dataset = MCQDataset(train_pairs)
val_dataset = MCQDataset(val_pairs)

### 4.1.5 - Data Collator

In [ ]:
# data collator
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    return_tensors="pt"
)

### 4.1.6 - Create Loader Objects

In [ ]:
train_loader =  DataLoader(
    train_dataset,
    batch_size = 16,
    shuffle=True,
    collate_fn=data_collator
)

val_loader = DataLoader(
    val_dataset,
    batch_size= 16,
    shuffle=False,
    collate_fn=data_collator
)

### 4.1.7 - LoadModel

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(

    MODEL_NAME,

    num_labels=2,

    dtype=torch.float32

)

model = model.cuda()

In [ ]:
model = model.to(device)
next(model.parameters()).device

### 4.1.8 - Define Optimizer

In [ ]:
# Optimizer
optimizer = AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

### 4.1.9 - Learning Rate + Scheduler

In [ ]:
# Learning rate Scheduler
epochs = 8

total_steps = len(train_loader) * epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

## 4.2 - Training Loop (1st stage)

In [ ]:
import wandb

wandb.init(

    project="24f1002705-DLGenAI-t12026",

    name="deberta-v3-base",

    config={

        "model": MODEL_NAME,
        "epochs": epochs,
        "batch_size": 16,
        "learning_rate": 2e-5,
        "max_length": 256,
        "optimizer": "AdamW",
        "scheduler": "Linear Warmup"

    }

)

In [ ]:
lowest_val_loss = 0

for epoch in range(epochs):

    ##################################
    # Training
    ##################################

    model.train()

    train_loss = 0

    for batch in train_loader:

        input_ids = batch["input_ids"].to(device)

        attention_mask = batch["attention_mask"].to(device)

        labels = batch["labels"].long().to(device)

        optimizer.zero_grad()

        outputs = model(

            input_ids=input_ids,

            attention_mask=attention_mask,

            labels=labels

        )

        loss = outputs.loss

        loss.backward()

        optimizer.step()

        scheduler.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    ##################################
    # Validation
    ##################################

    model.eval()

    val_loss = 0

    correct = 0

    total = 0

    with torch.no_grad():

        for batch in val_loader:

            input_ids = batch["input_ids"].to(device)

            attention_mask = batch["attention_mask"].to(device)

            labels = batch["labels"].long().to(device)

            outputs = model(

                input_ids=input_ids,

                attention_mask=attention_mask,

                labels=labels

            )

            val_loss += outputs.loss.item()

            predictions = torch.argmax(
                outputs.logits,
                dim=1
            )

            correct += (predictions == labels).sum().item()

            total += labels.size(0)

    val_loss /= len(val_loader)

    accuracy = correct / total

    ##################################
    # W&B Logging
    ##################################

    wandb.log({

        "epoch": epoch + 1,

        "train_loss": train_loss,

        "validation_loss": val_loss,

        "validation_accuracy": accuracy,

        "learning_rate": scheduler.get_last_lr()[0],

        "lowest validation loss": lowest_val_loss

    })

    ##################################
    # Console
    ##################################

    print(

        f"Epoch {epoch+1}/{epochs}"

        f" | Train Loss: {train_loss:.4f}"

        f" | Val Loss: {val_loss:.4f}"

        f" | Accuracy: {accuracy:.4f}"

    )

    ##################################
    # Save Best Model
    ##################################

    if val_loss <= lowest_val_loss:

        lowest_val_loss = val_loss

        model.save_pretrained("best_model")

        tokenizer.save_pretrained("best_model")

        artifact = wandb.Artifact(

            "best-deberta-model",

            type="model"

        )

        artifact.add_dir("best_model")

        wandb.log_artifact(artifact)

print(f"\nBest Validation Accuracy: {accuracy:.4f}")

wandb.finish()

### 4.2.2 - Save Checkpoint

In [ ]:
model.module.save_pretrained("deberta_stage1") if hasattr(model, "module") else model.save_pretrained("deberta_stage1")
tokenizer.save_pretrained("deberta_stage1")

### 4.2.3 - predict and create submission file

In [ ]:
predictions = []

for _, row in df_test.iterrows():

    ranking, _ = predict_mcq(

        model,
        tokenizer,
        row,
        device

    )

    predictions.append(

        " ".join(ranking[:3])

    )

In [ ]:
y_pred = pd.DataFrame({
    'ID': df_test['id'],
    'Prediction': [' '.join(pred) for pred in predictions]
})

y_pred.to_csv('submission_DaBERTa_v2.csv', index=False)

## 4.3 - Stage-2 Setup

In [ ]:
STAGE2_EPOCHS = 5
STAGE2_LR = 5e-6

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model = AutoModelForSequenceClassification.from_pretrained("deberta_stage1")
tokenizer = AutoTokenizer.from_pretrained("deberta_stage1")

model = model.to(device)

### 4.3.2 - Define stage2 optimizer + scheduler

In [ ]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

optimizer = AdamW(
    model.parameters(),
    lr=5e-6,
    weight_decay=0.01
)


total_steps = len(train_loader) * STAGE2_EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

## 4.4 - Training Loop (2nd Stage)

In [ ]:
wandb.init(

    project="24f1002705-t22026",

    name="deberta-stage2",

    config={

        "learning_rate": STAGE2_LR,
        "epochs": STAGE2_EPOCHS,
        "starting_checkpoint": "Stage1",
        "weight_decay": 0.01

    }

)

In [ ]:
lowest_val_loss = 0

for epoch in range(STAGE2_EPOCHS):

    ##################################
    # Training
    ##################################

    model.train()

    train_loss = 0

    for batch in train_loader:

        input_ids = batch["input_ids"].to(device)

        attention_mask = batch["attention_mask"].to(device)

        labels = batch["labels"].long().to(device)

        optimizer.zero_grad()

        outputs = model(

            input_ids=input_ids,

            attention_mask=attention_mask,

            labels=labels

        )

        loss = outputs.loss

        loss.backward()

        optimizer.step()

        scheduler.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    ##################################
    # Validation
    ##################################

    model.eval()

    val_loss = 0

    correct = 0

    total = 0

    with torch.no_grad():

        for batch in val_loader:

            input_ids = batch["input_ids"].to(device)

            attention_mask = batch["attention_mask"].to(device)

            labels = batch["labels"].long().to(device)

            outputs = model(

                input_ids=input_ids,

                attention_mask=attention_mask,

                labels=labels

            )

            val_loss += outputs.loss.item()

            predictions = torch.argmax(
                outputs.logits,
                dim=1
            )

            correct += (predictions == labels).sum().item()

            total += labels.size(0)

    val_loss /= len(val_loader)

    accuracy = correct / total

    ##################################
    # W&B Logging
    ##################################

    wandb.log({

        "epoch": epoch + 1,

        "train_loss": train_loss,

        "validation_loss": val_loss,

        "validation_accuracy": accuracy,

        "learning_rate": scheduler.get_last_lr()[0],

        "lowest validation loss": lowest_val_loss

    })

    ##################################
    # Console
    ##################################

    print(

        f"Epoch {epoch+1}/{epochs}"

        f" | Train Loss: {train_loss:.4f}"

        f" | Val Loss: {val_loss:.4f}"

        f" | Accuracy: {accuracy:.4f}"

    )

    ##################################
    # Save Best Model
    ##################################

    if val_loss <= lowest_val_loss:

        lowest_val_loss = val_loss

        model.save_pretrained("best_model")

        tokenizer.save_pretrained("best_model")

        artifact = wandb.Artifact(

            "best-deberta-model",

            type="model"

        )

        artifact.add_dir("best_model")

        wandb.log_artifact(artifact)

print(f"\nBest Validation Accuracy: {accuracy:.4f}")

wandb.finish()

### 4.4.2 - save checkpoint (stage 2)

In [ ]:
model.module.save_pretrained("deberta_stage2") if hasattr(model, "module") else model.save_pretrained("deberta_stage2")
tokenizer.save_pretrained("deberta_stage2")

## 4.5 - Predict and create Submissions

In [ ]:
predictions = []

for _, row in df_test.iterrows():

    ranking, _ = predict_mcq(

        model,
        tokenizer,
        row,
        device

    )

    predictions.append(

        " ".join(ranking[:3])

    )

In [ ]:
y_pred = pd.DataFrame({
    'ID': df_test['id'],
    'Prediction': [' '.join(pred) for pred in predictions]
})

y_pred.to_csv('submission_DaBERTa_v1_2stage.csv', index=False)

# 5 - Model - 3: DaBERTa 5-fold OOF ensembling

## 5.0 - Setup

In [ ]:
BATCH_SIZE = 16
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
FOLDS = 5
MODEL_NAME = "microsoft/deberta-v3-base"
STAGE1_EPOCHS = 8
STAGE2_EPOCHS = 5
MAX_LENGTH = 256

## 5.1 - Required Functions

### 5.1.1 - Process Data

combines some of the functionality of the beginning stages of Model 2, needed here due to multi fold processing

In [ ]:
# 3.4 - process data to prepare it for training
# takes in a dataframe and returns dataloader for training
def process_data(dataframe, tokenizer, shuffle):
    expanded_df = expand_dataset(dataframe)
    data_object = MCQDataset(expanded_df, tokenizer)

    # data collator
    data_collator = DataCollatorWithPadding(
        tokenizer=tokenizer,
        return_tensors="pt"
    )

    data_loader = DataLoader(
        data_object,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        collate_fn=data_collator,
        num_workers=2,
        pin_memory=True,
        persistent_workers=True
    )

    return data_loader

### 5.1.2 - Training Loop

In [ ]:
# 3.5 - the actual training loop
def train_model(
    model,
    tokenizer,
    train_loader,
    val_loader,
    optimizer,
    scheduler,
    epochs,
    save_dir
):
    lowest_val_loss = float("inf")
    best_accuracy = 0

    for epoch in range(epochs):
        
        # Training - put model into training mode
        model.train()

        train_loss = 0

        for batch in train_loader:
            # basically move the data to gpu for computation
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            # to convert them to floats32 for single precision calculations
            labels = batch["labels"].long().to(device)

            optimizer.zero_grad()

            outputs = model(
                input_ids = input_ids,
                attention_mask = attention_mask,
                labels=labels
            )
            
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            scheduler.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)

        # Evaluation
        model.eval()
        
        val_loss = 0

        correct = 0
    
        total = 0

        with torch.no_grad():
            for batch in val_loader:
                # basically move the data to gpu for computation
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                # to convert them to floats32 for single precision calculations
                labels = batch["labels"].long().to(device)
    
                outputs = model(
                    input_ids = input_ids,
                    attention_mask = attention_mask,
                    labels=labels
                )

                val_loss += outputs.loss.item()
                
                predictions = torch.argmax(
                    outputs.logits,
                    dim=1
                )
                
                correct += (predictions == labels).sum().item()

                total += labels.size(0)

        val_loss /= len(val_loader)

        accuracy = correct/total

        ##################################
        # Save Best Model
        ##################################
        if val_loss < lowest_val_loss:
            best_accuracy = accuracy
            
            lowest_val_loss = val_loss
    
            model.save_pretrained(save_dir)
    
            tokenizer.save_pretrained(save_dir)
    
            artifact = wandb.Artifact(
    
                save_dir,
    
                type="model"
    
            )
    
            artifact.add_dir(save_dir)
    
            wandb.log_artifact(artifact)

        
         ##################################
        # W&B Logging
        ##################################
    
        wandb.log({
    
            "epoch": epoch + 1,
    
            "train_loss": train_loss,
    
            "validation_loss": val_loss,
    
            "validation_accuracy": accuracy,
    
            "learning_rate": scheduler.get_last_lr()[0],
    
            "lowest validation loss": lowest_val_loss
    
        })
    
        ##################################
        # Console
        ##################################
    
        print(
    
            f"Epoch {epoch+1}/{epochs}"
    
            f" | Train Loss: {train_loss:.4f}"
    
            f" | Val Loss: {val_loss:.4f}"
    
            f" | Best Accuracy: {best_accuracy:.4f}"
    
        )
    
    print(f"\nLowest Validation Loss: {lowest_val_loss:.4f}")

    return {
        'loss': lowest_val_loss,
        'accuracy': accuracy,
        'checkpoint': save_dir
    }

### 5.1.3 - Clear Data

In [ ]:
# 3.6 - clean up between stages and folds
def clear_data(model, optimizer, scheduler):
    try:
        del model
        del optimizer
        del scheduler

        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        gc.collect()
    
    except Exception as e:
        return("could not clean up model, optimizer and scheduler", e)

    return

## 5.2 - Main function

In [ ]:
def daberta_2_stage():
    df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

    kf = StratifiedKFold(
        n_splits=FOLDS,
        shuffle=True,
        random_state=SEED
    )

    for fold, (train_idx, val_idx) in enumerate(
        kf.split(df, df['answer']),
        start=1
    ):
        print(f"\n========== Fold {fold} ==========\n")

        train_df = df.iloc[train_idx].reset_index(drop=True)
        val_df = df.iloc[val_idx].reset_index(drop=True)
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME,
            num_labels=2,
            dtype=torch.float32
        )

        model = model.to(device)

        train_loader = process_data(
            train_df,
            tokenizer,
            True
        )

        val_loader = process_data(
            val_df,
            tokenizer,
            False
        )

        # Stage - 1 
        # Setup
        # optimizer
        optimizer1 = AdamW(
            model.parameters(),
            lr=5e-6,
            weight_decay=0.01
        )

        # scheduler
        scheduler1 = get_linear_schedule_with_warmup(
            optimizer1,
            num_warmup_steps=int(0.1 * (len(train_loader) * STAGE1_EPOCHS)),
            num_training_steps=(len(train_loader) * STAGE1_EPOCHS)
        )

        wandb.init(

            project="24f1002705-DLGenAI-t12026",
        
            name=f"deberta-v3-base-fold{fold}_stage1",
        
            config={
        
                "model": MODEL_NAME,
                "epochs": STAGE1_EPOCHS,
                "batch_size": 16,
                "learning_rate": 5e-6,
                "max_length": MAX_LENGTH,
                "optimizer": "AdamW",
                "scheduler": "Linear Warmup"
        
            }
        
        )

        # trainer
        train_model(
            model,
            tokenizer,
            train_loader,
            val_loader,
            optimizer1,
            scheduler1,
            STAGE1_EPOCHS,
            f"best_model_fold{fold}_stage1"
        )

        wandb.finish()

        # interstage clean up
        clear_data(model, optimizer1, scheduler1)

## 5.3 - Execute

In [ ]:
daberta_2_stage()

## 5.4 - Inference

In [ ]:
# 3.7 - get models
def get_model_paths():
    folds = [1,2,3,5]
    paths = []
    for fold in folds:
        paths.append(f"/kaggle/input/models/siddharthavarma12/smart-mcq-deberta/pytorch/fold{fold}/1/best_model_fold{fold}_stage2")
    return paths


# 3.8 - predict for one model
def predict_model(
    test_loader,
    model
):
    model.eval()

    logits = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)

            attention_mask = batch["attention_mask"].to(device)

            outputs = model(

                input_ids=input_ids,

                attention_mask=attention_mask

            )

            positive_logits = outputs.logits[:,1]

            logits.extend(

                positive_logits.cpu().numpy()

            )

    logits = np.array(logits)

    logits = logits.reshape(-1,5)

    clear_data(model)

    return logits

def ensemble_predictions(
    test_loader
):
    predictions = []
    
    for path in get_model_paths():
        model = AutoModelForSequenceClassification.from_pretrained(
            path
        )
        model = model.to(device)
        pred = predict_model(test_loader, model)
        predictions.append(pred)

    predictions = np.stack(
        predictions,
        axis=0
    )

    predictions = np.mean(
        predictions,
        axis=0
    )

    return predictions

def create_submission(predictions, df):
    options = np.array([
        "A",

        "B",

        "C",

        "D",

        "E"]
    )

    final_predictions = []

    for row in predictions:
        ranking = np.argsort(
            -row
        )

        prediction = " ".join(

            options[ranking[:3]]

        )
        
        final_predictions.append(prediction)

    submission = pd.DataFrame({
        "id": df["id"],
        "prediction": final_predictions        
    })

    submission.to_csv(
        "submission.csv",
        index=False
    )

    return submission

# ____________________________________________________________________________________________________
# ____________________________________________________________________________________________________
# ____________________________________________________________________________________________________
def main():
    df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    test_loader = process_data(
        df,
        tokenizer,
        False
    )

    predictions = ensemble_predictions(test_loader)

    submission = create_submission(
        predictions,
        df
    )
    submission.head()
    
if __name__ == "__main__":

    main()
